# RoundScoreCard 一致性校验示例

本示例演示如何使用 `RoundScoreCard(decimal=1)`，并验证以下一致性：

- `scorecard_points()` 输出的基础分与各特征子分已按 `decimal` 调整
- 基于 `scorecard_points()` 逐条回算的总分，与 `predict()` / `predict_score()` 一致
- `get_detailed_score()` 中的分箱标签与子分，与评分卡表一致
- `export_deployment_code()` 生成的部署代码结果，与模型预测一致


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from hscredit.core.binning import OptimalBinning
from hscredit.core.models import RoundScoreCard
from hscredit.utils.datasets import germancredit


In [2]:
df = germancredit().copy()
y = df['class'].astype(int)
X = df.drop(columns=['class'])

X_train, X_test, y_train, _ = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

binner = OptimalBinning(method='target_bad_rate', max_n_bins=5)
binner.fit(X_train, y_train)

scorecard = RoundScoreCard(
    pdo=60,
    rate=2,
    base_odds=35,
    base_score=750,
    direction='ascending',
    decimal=1,
    binner=binner,
)
scorecard.fit(binner.transform(X_train, metric='woe'), y_train, input_type='woe')

sample = X_test.iloc[:60].copy()
points = scorecard.scorecard_points()
points.head()


,变量名称,变量含义,变量分箱,对应分数,WOE值
0,基础分,截距项（基准分数）,-,984.6,NaN
1,status_of_existing_checking_account,,no checking account,-80.0,-1.1841
2,status_of_existing_checking_account,,... >= 200 DM / salary assignments for at leas...,-9.0,-0.1335
3,status_of_existing_checking_account,,0 <= ... < 200 DM,28.0,0.4152
4,status_of_existing_checking_account,,... < 0 DM,51.6,0.7639


In [3]:
def manual_score_from_points(scorecard, X_raw):
    points = scorecard.scorecard_points()
    base_score = float(points.loc[points['变量名称'] == '基础分', '对应分数'].iloc[0])
    special_codes = scorecard._get_deployment_special_codes()

    total = np.full(len(X_raw), base_score, dtype=float)
    feature_scores = {}
    matched_bins = {}

    for feature in scorecard.feature_names_:
        feature_points = points[points['变量名称'] == feature].reset_index(drop=True)
        feature_score_values = []
        feature_bin_labels = []

        for value in X_raw[feature].tolist():
            matched_score = None
            matched_label = None
            for _, row in feature_points.iterrows():
                bin_label = row['变量分箱']
                condition = scorecard._bin_label_to_python_condition('value', bin_label, special_codes=special_codes)
                matched = eval(condition, {'pd': pd}, {'value': value})

                if not matched and isinstance(bin_label, str) and ',' in bin_label and not bin_label.strip().startswith(('[', '(')):
                    categories = [item.strip() for item in bin_label.split(',') if item.strip()]
                    matched = str(value) in categories

                if matched:
                    matched_score = float(row['对应分数'])
                    matched_label = bin_label
                    break

            if matched_score is None:
                fallback_bins = [
                    (row['变量分箱'], float(row['对应分数']))
                    for _, row in feature_points.iterrows()
                ]
                matched_score = float(scorecard._get_deployment_default_score(fallback_bins))
                for label, score in fallback_bins:
                    if float(score) == matched_score and not scorecard._is_missing_descriptor(label) and not scorecard._is_special_descriptor(label):
                        matched_label = label
                        break

            if matched_score is None or matched_label is None:
                raise KeyError(f'未能在 scorecard_points 中找到特征 {feature} 值 {value!r} 的匹配分箱')

            feature_score_values.append(matched_score)
            feature_bin_labels.append(matched_label)

        mapped = np.asarray(feature_score_values, dtype=float)
        feature_scores[feature] = mapped
        matched_bins[feature] = pd.Series(feature_bin_labels, index=X_raw.index)
        total += mapped

    total = np.round(total, scorecard.decimal)
    return total, feature_scores, matched_bins

manual_scores, feature_scores, matched_bins = manual_score_from_points(scorecard, sample)
predicted = scorecard.predict(sample, input_type='raw')
predicted_by_score = scorecard.predict_score(sample, input_type='raw')

np.testing.assert_allclose(predicted, manual_scores, atol=1e-9)
np.testing.assert_allclose(predicted, predicted_by_score, atol=1e-9)
print('scorecard_points 与 predict/predict_score 一致性校验通过')


scorecard_points 与 predict/predict_score 一致性校验通过


In [4]:
detailed = scorecard.get_detailed_score(sample, include_reason=True)
np.testing.assert_allclose(detailed[('样本信息', '总分')].to_numpy(dtype=float), predicted, atol=1e-9)

for feature in scorecard.feature_names_:
    np.testing.assert_allclose(
        detailed[(feature, '分数')].to_numpy(dtype=float),
        feature_scores[feature],
        atol=1e-9,
    )
    detailed_bins = detailed[(feature, '分箱')].astype(str).map(scorecard._normalize_rule_label)
    manual_bins = matched_bins[feature].astype(str).map(scorecard._normalize_rule_label)
    assert detailed_bins.tolist() == manual_bins.tolist()

namespace = {}
exec(scorecard.export_deployment_code(language='python'), namespace)
deployed_scores = sample.apply(lambda row: namespace['calculate_score'](row.to_dict()), axis=1).to_numpy(dtype=float)
np.testing.assert_allclose(predicted, deployed_scores, atol=1e-9)

print('detailed_score 与部署代码一致性校验通过')
detailed.head()


detailed_score 与部署代码一致性校验通过


样本信息                               status_of_existing_checking_account  \
  样本索引      总分   截距分数                                                原始值   
0  115   759.2  984.6                                no checking account   
1  346   956.2  984.6                                  0 <= ... < 200 DM   
2  328   987.4  984.6  ... >= 200 DM / salary assignments for at leas...   
3  974   890.3  984.6                                no checking account   
4  587  1042.8  984.6                                         ... < 0 DM   

                                                                    \
                                                  分箱     WOE    分数   
0                                no checking account -1.1841 -80.0   
1                                  0 <= ... < 200 DM  0.4152  28.0   
2  ... >= 200 DM / salary assignments for at leas... -0.1335  -9.0   
3                                no checking account -1.1841 -80.0   
4                                         ... < 0 DM  0.7639  51.6   

  duration_in_month                        ...  \
                原始值            分箱     WOE  ...   
0                48  [30.0, 51.0)  0.5304  ...   
1                13   [6.5, 15.0) -0.3441  ...   
2                36  [30.0, 51.0)  0.5304  ...   
3                30  [30.0, 51.0)  0.5304  ...   
4                12   [6.5, 15.0) -0.3441  ...   

  number_of_people_being_liable_to_provide_maintenance_for  \
                                                        分数   
0                                                0.0         
1                                                0.0         
2                                                0.0         
3                                                0.0         
4                                                0.0         

                                  telephone  \
                                        原始值   
0  yes, registered under the customers name   
1                                      none   
2                                      none   
3  yes, registered under the customers name   
4                                      none   

                                                         foreign_worker       \
                                         分箱     WOE   分数            原始值   分箱   
0  yes, registered under the customers name -0.0665 -2.6            yes  yes   
1                                      none  0.0442  1.8            yes  yes   
2                                      none  0.0442  1.8            yes  yes   
3  yes, registered under the customers name -0.0665 -2.6            yes  yes   
4                                      none  0.0442  1.8            yes  yes   

                                                            评分分析  
     WOE   分数                                               评分原因  
0  0.045  3.2  status_of_existing_checking_account拉低108.0分(当前...  
1  0.045  3.2  purpose拉低47.5分(当前-32.7分); present_employment_s...  
2  0.045  3.2  purpose拉低47.5分(当前-32.7分); status_of_existing_c...  
3  0.045  3.2  status_of_existing_checking_account拉低108.0分(当前...  
4  0.045  3.2  duration_in_month拉低31.2分(当前-24.1分); other_debt...  

[5 rows x 84 columns]